# A0. Custom mechanisms

In this tutorial, we dive into the nitty gritty of how membrane mechanisms are implemented in Dendra. We will explore how to define custom ion channels / dynamics and synaptic mechanisms, and how to integrate them into your neuronal models.



## Mechanism vs State

Dendra separates the *Mechanism* (user-facing membrane mechanism) from the *State* helper used to declare per-compartment state variables and their dynamics.

- **Mechanism** (`Mechanism`): declares parameters, states, ionic usage, currents, and lifecycle hooks.
- **State** (`mechanisms.State`): inner helper class where you declare state variables and their ODEs/kinetics; register with `Mechanism.STATE(...)`.


## Declaring state variables and dynamics

Use a `State` subclass to name state variables and define their dynamics with `DERIVATIVE` or `KINETIC`. `breakpoint` computes `ASSIGNED` variables (steady states, time constants, etc.) that appear in derivatives/currents but are not themselves state or parameter variables.


In [ ]:
from dendra.models.mechanisms import Mechanism as M
from dendra.models.mechanisms import State as S
from dendra.models.mechanisms.ops import exp, vtrap

class mhn(S):
    has_q10 = True  # optional Q10 scaling hook

    S.STATE("m", "h", "n")
    S.METHOD("cnexp")
    S.DERIVATIVE(
        "m' = (minf - m) / mtau",
        "h' = (hinf - h) / htau",
        "n' = (ninf - n) / ntau",
    )
    S.ASSIGNED("minf", "mtau", "hinf", "htau", "ninf", "ntau")

    def calc_q10(self):
        return 3.0 ** ((self.celsius - 6.3) / 10.0)

    def breakpoint(self, v, states=None):
        q10 = self.q10()
        alpha_m = 0.1 * vtrap(-(v + 40), 10)
        beta_m = 4 * exp(-(v + 65) / 18)
        tot = alpha_m + beta_m
        mtau = 1 / (q10 * tot)
        minf = alpha_m / tot

        alpha_h = 0.07 * exp(-(v + 65) / 20)
        beta_h = 1 / (exp(-(v + 35) / 10) + 1)
        tot = alpha_h + beta_h
        htau = 1 / (q10 * tot)
        hinf = alpha_h / tot

        alpha_n = 0.01 * vtrap(-(v + 55), 10)
        beta_n = 0.125 * exp(-(v + 65) / 80)
        tot = alpha_n + beta_n
        ntau = 1 / (q10 * tot)
        ninf = alpha_n / tot

        # return dict whose keys match ASSIGNED names
        return {"mtau": mtau, "minf": minf, "htau": htau, "hinf": hinf, "ntau": ntau, "ninf": ninf}

    def inf(self, v):
        states = self.breakpoint(v, None)
        return {"m": states["minf"], "h": states["hinf"], "n": states["ninf"]}


class hh(M):
    """
    Classic Hodgkin-Huxley mechanism with m/h/n gates and leak.
    """

    M.STATE(mhn)
    M.GLOBAL(gnabar=0.12, gkbar=0.036, gl=0.0003, ena=50.0, ek=-77.0, el=-54.3)
    M.NONSPECIFIC_CURRENT("il", "ina", "ik")
    # Example for ions: M.USEION("na", read=("nai", "nao", "ena"), write=("ina",))

    def il(self, v):
        return self.gl * (v - self.el)

    def ina(self, v):
        gna = self.gnabar * self.m**3 * self.h
        return gna * (v - self.ena)

    def ik(self, v):
        gk = self.gkbar * self.n**4
        return gk * (v - self.ek)


## Methods of integration

Dendra supports multiple methods for integrating `State` dynamics. The preferred way to select an integration method is with the `State.METHOD(...)` declaration inside a `State` subclass:

```python
from dendra.models.mechanisms._state import State as S

class MyState(S):
    S.STATE("x", "y")
    S.METHOD("rosenbrock", fallback="derivimplicit")

    S.DERIVATIVE(
        "x' = -a*x + y*y",
        "y' = -b*y + x",
    )
```

`S.METHOD(method, **kwargs)` selects the integration method and passes method-specific options to the solver builder. The older class-attribute form is still supported for backward compatibility:

```python
class MyState(S):
    method = "derivimplicit"
```

However, `S.METHOD(...)` is preferred because it also supports method options such as iteration counts, damping, fallbacks, or solver-specific declarations.


### `"cnexp"`

Exact exponential integration for independent scalar affine ODEs.

Use this for equations of the form:

```text
x' = a + b*x
```

or, equivalently, common gating equations such as:

```text
x' = (x_inf - x) / tau_x
```

where coefficients are treated as fixed over the timestep.

Example:

```python
class Gate(S):
    S.STATE("m")
    S.METHOD("cnexp")

    S.DERIVATIVE(
        "m' = (minf - m) / mtau",
    )
```

`"cnexp"` is the default method. It is usually the fastest and most accurate method for Hodgkin-Huxley-style independent gates. It should be preferred whenever the state equation is independent and affine in its own state.

`cnexp` can also be applied to systems where each scalar equation is affine in its own state but depends on other state variables. In that case, the other states are treated as constants over the timestep, using their values at the beginning of the step. This is a frozen-coupling approximation: it is fast and often useful, but it is not an exact coupled solve.

Supported aliases include:

```text
"cnexp"
"rush_larsen"
"rl"
```


### `"bufferimplicit"`

Specialized implicit integration for two-state free/bound buffer systems.

Use this for systems like calcium buffering:

```text
bound' = ku * free * (1 - bound) - kr * bound
free'  = source - gamma * bound'
```

For example:

```python
class oc_cai(S):
    S.STATE("oc", "cai")
    S.METHOD("bufferimplicit", bound="oc", free="cai")

    S.DERIVATIVE(
        "oc' = ku * cai * (1 - oc) - kr * oc",
        "cai' = -ica * SA / Vol / (2*FARADAY) - nb * Bi * (ku * cai * (1 - oc) - kr * oc)",
    )
```

This method performs a closed-form backward-Euler update by reducing the implicit system to a scalar quadratic solve. It avoids Newton iteration, preserves the buffer/free-concentration conservation relation under the backward-Euler discretization, and is much faster than `derivimplicit` for this motif.

This is the preferred method for `oc/cai`-style intracellular calcium buffering systems.

Common options:

```python
S.METHOD(
    "bufferimplicit",
    bound="oc",
    free="cai",
    clamp_discriminant=True,
    eps=1e-30,
    clamp_bound=False,
    bound_min=0.0,
    bound_max=1.0,
    free_min=None,
    strict=True,
    fallback=None,
)
```

Supported aliases include:

```text
"bufferimplicit"
"buffer"
"implicit_buffer"
"ca_buffer"
"calcium_buffer"
"calciumbuffer"
```


### `"linearimplicit"` / `"sparse"`

Implicit backward-Euler integration for coupled affine-linear systems.

Use this for systems of the form:

```text
x' = A*x + b
```

where the state variables may be coupled, but the right-hand side is affine-linear in the state vector.

Example:

```python
class Markov2(S):
    S.STATE("C", "O")
    S.METHOD("sparse")

    S.DERIVATIVE(
        "C' = -alpha*C + beta*O",
        "O' =  alpha*C - beta*O",
    )
```

The method generates the backward-Euler update:

```text
(I - dt*A) * x_next = x + dt*b
```

This is the correct choice for coupled linear Markov models, linear kinetic schemes, linear exchange systems, and other affine-coupled state systems.

Although `"sparse"` is provided as an alias for NMODL familiarity, the current implementation is best thought of as a generated batched small-system linear implicit solver. It does not require the user to manually assemble a sparse matrix.

This method is not valid for nonlinear state coupling such as:

```text
x' = x*y
x' = sin(x) - y
oc' = ku*cai*(1 - oc) - kr*oc
```

For the last case, use `"bufferimplicit"` if the equations match the calcium-buffer motif, or `"rosenbrock"` / `"derivimplicit"` otherwise.

Common options:

```python
S.METHOD(
    "sparse",
    solver="auto",
    jacobian_regularization=0.0,
    strict=True,
    fallback=None,
)
```

Fallback examples:

```python
S.METHOD("sparse", fallback="rosenbrock")
S.METHOD("sparse", fallback="derivimplicit")
```

Supported aliases include:

```text
"linearimplicit"
"linear_implicit"
"implicitlinear"
"implicit_linear"
"affineimplicit"
"affine_implicit"
"sparse"
"be_linear"
"backward_euler_linear"
"linear_be"
```


### `"rosenbrock"`

A general linearly implicit method for nonlinear ODE systems.

Given:

```text
x' = f(x)
```

Rosenbrock-Euler evaluates the RHS and Jacobian at the beginning of the timestep, then solves:

```text
(I - gamma*dt*J) * k = dt*f(x)
x_next = x + damping*k
```

Example:

```python
class NonlinearState(S):
    S.STATE("x", "y")
    S.METHOD("rosenbrock", fallback="derivimplicit")

    S.DERIVATIVE(
        "x' = -a*x + y*y",
        "y' = -b*y + x",
    )
```

This is a good general-purpose method for nonlinear systems when `cnexp`, `bufferimplicit`, and `linearimplicit` do not apply. It is usually much faster than `derivimplicit` because it uses one RHS evaluation, one Jacobian evaluation, and one linear solve, without Newton iterations or line search.

For affine-linear systems, `"rosenbrock"` with `gamma=1.0` gives the same update as backward Euler. For nonlinear systems, it is an approximation to fully implicit backward Euler based on a local linearization.

Common options:

```python
S.METHOD(
    "rosenbrock",
    gamma=1.0,
    damping=1.0,
    jacobian_regularization=0.0,
    strict=True,
    fallback=None,
)
```

Supported aliases include:

```text
"rosenbrock"
"rosenbrock1"
"rosenbrock_euler"
"linearlyimplicit"
"linearly_implicit"
"linearized_implicit"
"linearly_implicit_euler"
"linimplicit"
"semiimplicit"
"semi_implicit"
```


### `"derivimplicit"`

Fully implicit backward-Euler integration for general ODE systems using Newton-Raphson iteration.

Use this for nonlinear and coupled ODE systems when a more specialized method is not available or when a fully converged implicit solve is desired.

Example:

```python
class GeneralImplicitState(S):
    S.STATE("x", "y")
    S.METHOD(
        "derivimplicit",
        newton_iters=4,
        damping=1.0,
        line_search=False,
    )

    S.DERIVATIVE(
        "x' = -a*x + y*y",
        "y' = -b*y + sin(x)",
    )
```

`derivimplicit` constructs the backward-Euler residual:

```text
R(x_next) = x_next - x - dt*f(x_next)
```

and approximately solves:

```text
R(x_next) = 0
```

using Newton iterations.

This is the most general method, but it is also the most expensive. It may require multiple RHS evaluations, Jacobian evaluations, and linear solves per timestep. It can also have convergence issues for very stiff systems, poor initial guesses, or large timesteps.

Dendra attempts to generate analytic Jacobians when possible. If analytic Jacobian generation fails, the method may fall back to a slower Jacobian path, depending on solver options.

Common options:

```python
S.METHOD(
    "derivimplicit",
    max_iter=8,
    tol=1e-8,
    damping=1.0,
    line_search=True,
    max_ls_steps=8,
    ls_decay=0.5,
    jacobian_regularization=0.0,
    detach_newton=None,
    create_graph=None,
)
```

Aliases for `max_iter` include:

```text
"newton_iters"
"iterations"
```

For compiled training workloads, a fixed small number of Newton iterations with line search disabled is often more compiler-friendly:

```python
S.METHOD(
    "derivimplicit",
    newton_iters=3,
    damping=1.0,
    line_search=False,
)
```

Supported aliases include:

```text
"derivimplicit"
"deriv_implicit"
"implicit"
"backward_euler"
"be"
```


### Choosing a method

A useful rule of thumb is:

- Use "cnexp" when each state equation is independent and affine in its own state.

- Use "bufferimplicit" for free/bound buffer systems such as oc/cai calcium buffering.

- Use "linearimplicit" or "sparse" for coupled affine-linear systems, such as Markov state models.

- Use "rosenbrock" for nonlinear systems when you want a fast, stable, fixed-work method.

- Use "derivimplicit" when you need the most general fully implicit solver or as a reference/fallback.

For example:

```python
# Independent HH-style gate
S.METHOD("cnexp")

# Calcium buffer motif
S.METHOD("bufferimplicit", bound="oc", free="cai")

# Coupled linear Markov model
S.METHOD("sparse")

# General nonlinear fast fallback
S.METHOD("rosenbrock", fallback="derivimplicit")

# Fully implicit Newton solve
S.METHOD("derivimplicit", newton_iters=4, line_search=False)
```

A recommended fallback pattern for nonlinear mechanisms is:

```python
S.METHOD("rosenbrock", fallback="derivimplicit")
```

A recommended fallback pattern for systems expected to be linear is:

```python
S.METHOD("sparse", fallback="rosenbrock")
```

This lets Dendra use the fastest valid solver when the symbolic structure matches, while retaining a more general method when it does not.

## Q10 scaling
- Set `has_q10 = True` on a `State` subclass to enable temperature scaling. Dendra then exposes `self.q10()` in the state, calling `calc_q10()` if provided (otherwise defaulting to 1).
- Implement `calc_q10(self)` to return the multiplicative factor as a function of `self.celsius`. In the HH gates above:
  ```python
  def calc_q10(self):
      return 3.0 ** ((self.celsius - 6.3) / 10.0)
  ```
- Use `self.q10()` inside `breakpoint` or kinetic rate calculations to scale time constants and rates.


## State `inf` helper
- Implement `inf(self, v)` on a `State` to return a mapping of steady-state values for the declared state variables (e.g., `{"m": minf, "h": hinf}`) at a given voltage. Dendra uses this during initialization to set state variables unless overridden.
- You can also use `inf` to encode any custom initialization rule you want; it’s invoked during `initialize()` for states.


## Multiple State bundles per Mechanism
A mechanism can register multiple independent `State` subclasses. For example, the HH gates `m`, `h`, and `n` could each be defined as separate `State` classes and then registered via multiple `M.STATE(...)` calls (or a single call with multiple states). Dendra will aggregate all declared states/derivatives and step them together.



## Buffers and `initial`
- `BUFFER(name, ...)` declares auxiliary per-compartment buffers stored on the state (not evolved by an ODE). They are useful for geometry- or configuration-dependent terms reused in derivatives.
- Implement `initial(self, v)` on a `State` to populate these buffers (and optionally states) at initialization. It runs once during `initialize()` and can depend on morphology (e.g., `self.diam`).
- Example (extracellular Ca accumulation in `caextscale`):
  ```python
  class cao(S):
      S.STATE("cao")
      S.GLOBAL(lseg=1.0, txfer=4511.0, FARADAY=96500.0, cabath=2.0, fhspace=1.0)
      S.BUFFER("SA", "Vol_peri")
      S.DERIVATIVE("cao' = (ica*SA/(2*Vol_peri*FARADAY) + (cabath - cao)/txfer)")

      def initial(self, v):
          self.SA = math.pi * (1e-4) * self.diam * self.lseg
          Vol = math.pi * ((1e-4) * (self.diam / 2)) ** 2 * self.lseg
          self.Vol_peri = (
              math.pi * ((1e-4) * ((self.diam + self.fhspace) / 2)) ** 2 * self.lseg
          ) - Vol
  ```
  `SA` and `Vol_peri` are computed once, then reused inside the derivative during simulation.


## API overview
### State classmethods (at `State` definition)
- `STATE(name, ...)`: declare state variables.
- `DERIVATIVE("x' = expr", ...)`: symbolic ODEs for states.
- `KINETIC("~ a <-> b (alpha, beta)", ...)`: Markov/kinetic schemes between states.
- `ASSIGNED(name, ...)`: computed per-compartment variables; set in `breakpoint`.
- `GLOBAL` / `RANGE`: shared vs per-compartment parameters scoped to the State.
- `BUFFER(name, ...)`: declare auxiliary per-compartment buffers initialized in `initial` and reused in derivatives/assigned calculations.


### State lifecycle hooks
- `initial(self, v)`: optional initializer run during `initialize()` to set buffers/states (e.g., geometry-dependent buffers declared via `BUFFER`).
- `breakpoint(self, v, states=None)`: compute `ASSIGNED`/intermediate values each step before derivatives/currents are evaluated; return a dict matching `ASSIGNED` names if you choose to return values.
- `inf(self, v)`: return steady-state values for state variables (used for initialization unless overridden by the caller).
- `calc_q10(self)`: optional helper when `has_q10=True`; returns the temperature scaling factor consumed by `self.q10()` inside your kinetics.


### Mechanism classmethods (at mechanism definition)
- `STATE(StateSubclass, ...)`: register State bundles used by the mechanism.
- `GLOBAL`, `RANGE`: shared vs per-compartment parameters on the mechanism.
- `ASSIGNED(name, ...)`: per-compartment buffers/computed vars (analogous to `BUFFER` in State); typically set in `initial`/`breakpoint`.
- `INIT(...)`: initialization hooks for mechanism buffers/states.
- `USEION(ion, read=(), write=())`: declare ionic dependencies. `read`/`write` must be in `{ion}i`, `{ion}o`, `e{ion}`, `i{ion}` (e.g., read `("ena", "nai", "nao")`, write `("ina",)` to contribute Na current).
- `NONSPECIFIC_CURRENT(name, ...)`: register membrane currents not tied to an ion.
- `EXPLICIT(name, ...)`: mark voltage-independent currents when assembling conductance terms.



### Mechanism lifecycle hooks
- `initial(self, v)`: optional initializer run during `initialize()`; set mechanism-level `ASSIGNED` buffers or other cached quantities.
- `breakpoint(self, v)`: compute mechanism currents and mechanism-level `ASSIGNED` values each step (called before current assembly).
- Current methods (`ina`, `ik`, `il`, etc.): implement currents declared via `NONSPECIFIC_CURRENT` or ion writes in `USEION`. These are invoked to assemble total membrane/ionic currents.



**Example (h-channel with mechanism-level ASSIGNED):**
```python
class h(M):
    M.STATE(s, f)
    M.GLOBAL(gbar=0.0001)
    M.USEION("k", read=["ek"], write=["ik"])
    M.USEION("na", read=["ena"], write=["ina"])
    M.ASSIGNED("g")  # mechanism-level buffer, set in initial/breakpoint

    def initial(self, v):
        self.breakpoint(v)  # compute g at t=0

    def breakpoint(self, v):
        self.g = self.gbar * (0.5 * self.s + 0.5 * self.f)

    def ina(self, v):
        return 0.5 * self.g * (v - self.ena)

    def ik(self, v):
        return 0.5 * self.g * (v - self.ek)
```
Here `ASSIGNED("g")` acts like a mechanism-level buffer (analogous to `BUFFER` in State) that is populated in `initial`/`breakpoint` and reused in current computations.


## Ions
Dendra has built-in support for ionic concentrations and currents via the `USEION` mechanism classmethod. This allows you to read/write ionic variables (e.g., `nai`, `ek`) and have Dendra automatically handle the underlying ion dynamics and current assembly. You can also define custom ion dynamics by declaring a `State` bundle that declares the relevant ion dynamics.

By default, Dendra includes built-in support for the following ions and their associated variables:

- Sodium (Na): `nai`, `nao`, `ena`, `ina`
- Potassium (K): `ki`, `ko`, `ek`, `ik`
- Calcium (Ca): `cai`, `cao`, `eca`, `ica`

You can also define custom ions by using the `register_ion` function to specify the ion name, valence, equilibrium potential, and initial concentrations. For example, to define a custom chloride ion:

```python
from dendra.models.mechanisms import register_ion
register_ion(
    name="cl",
    valence=-1,
    e=-65.0,
    i0=10.0, # intracellular concentration [mM]
    o0=110.0, # extracellular concentration [mM]
)
```
This will allow you to use "cl" in `USEION` declarations and have Dendra automatically handle the associated variables (`cli`, `clo`, `ecl`, `icl`) and dynamics.

### USEION read/write semantics

Fundamentally, `USEION` is a way to declare that your mechanism interacts with certain ionic variables. The `read` variables are those that your mechanism will read (e.g., reversal potentials, concentrations), and the `write` variables are those that your mechanism will contribute to or update (e.g., ionic currents, ion concentrations).

For example, when you declare `Mechanism.USEION("na", read=["ena", "nai", "nao"], write=["ina"])` in your `Mechanism` definition, you are telling Dendra that this mechanism will read the sodium reversal potential (`ena`) and concentrations (`nai`, `nao`), and will contribute to the sodium current (`ina`). Dendra will then automatically include these variables in the appropriate places:

- `ena`, `nai`, `nao` will be available as variables that you can use in your `breakpoint` or current methods. They are also visible to any `State` subclasses registered to the mechanism, so you can use them in your state dynamics as well.
- `ina` will be expected as a *method* on your mechanism that accepts the membrane voltage as an argument and computes the sodium current contribution based on the voltage, and any declared read variables and any state variables or parameters you have.

"`<ion>i` and `<ion>o` can also be written to by mechanisms if you want to implement custom ion dynamics (e.g., intracellular calcium accumulation). In that case, Dendra will automatically include the appropriate terms in the ion concentration updates during simulation. For example, if you declare `USEION("ca", read=["ica"], write=["cai"])`, "ica" will be available as a variable you can use in your `DERIVATIVE` definitions.

Example 1: intracellular Ca2+ accumulation:

```python
import math

from dendra.models.mechanisms._mechanism import Mechanism as M
from dendra.models.mechanisms._state import State as S
from dendra.models.mechanisms.ops import *


class oc_cai(S):
    S.STATE("oc", "cai")
    S.GLOBAL_SIGNED(lseg=1e-3, ku=100.0, kr=0.238, nb=4.0, Bi=0.001, FARADAY=96500.0)
    S.BUFFER("SA", "Vol")

    S.DERIVATIVE(
        "oc' = ku * cai * (1-oc) - kr * oc",
        "cai' = (-ica * (SA) / Vol / (2*FARADAY) - (nb * Bi * (ku * cai * (1 - oc) - kr * oc)))",
    )

    def initial(self, v):
        self.SA = math.pi * (1e-4) * self.diam * self.lseg
        self.Vol = math.pi * ((1e-4) * (self.diam / 2)) ** 2 * self.lseg


class caintscale(M):
    M.STATE(oc_cai)
    M.USEION("ca", read=["ica"], write=["cai"])
    M.INIT(oc=0.05)
```

How this works:
- The `oc_cai` state bundle declares two state variables: `oc` (open channel fraction) and `cai` (intracellular calcium concentration). It also declares some global parameters and buffers for geometry-dependent terms.
- The `DERIVATIVE` definitions specify the ODEs for `oc` and `cai`. Note that the `cai` derivative depends on the calcium current `ica`, which is declared as a read variable in the `USEION` statement of the `caintscale` mechanism.
- The `initial` method computes the geometry-dependent buffers `SA` and `Vol` at initialization based on the compartment diameter.
- The `caintscale` mechanism registers the `oc_cai` state bundle and declares that it reads the `ica` current and writes to the `cai` concentration. Dendra will automatically handle the updates to `cai` during simulation based on the declared dynamics. 

```{important}
As a rule, only one mechanism should write to a given ionic concentration (e.g., `cai`) to avoid conflicts, but multiple mechanisms can read the same ion variables.
```

Example 2: if you want to update the Hodgkin-Huxley mechanism to interact with ions:

```python
from dendra.models.mod.HH import mhn
from dendra.models.mechanism import Mechanism as M

class hh(M):
    M.STATE(mhn)
    M.GLOBAL(gnabar=0.12, gkbar=0.036, gl=0.0003, el=-54.3)
    M.USEION("na", read=["ena"], write=["ina"])
    M.USEION("k", read=["ek"], write=["ik"])
    M.NONSPECIFIC_CURRENT("il")

    def ina(self, v):
        return self.gnabar * (self.m ** 3) * self.h * (v - self.ena)

    def ik(self, v):
        return self.gkbar * (self.n ** 4) * (v - self.ek)

    def il(self, v):
        return self.gl * (v - self.el)
```


### Per-population ion style flags, concentrations, and equilibria

Initialization of ion concentrations and reversal potentials are handled by the `equilibria` and `concentrations` methods of the `Population` class. By default, Dendra will initialize ion concentrations and equilibria to their default values:

- Intracellular concentrations: `nai=10 mM`, `ki=54.4 mM`, `cai=5e-5 mM`
- Extracellular concentrations: `nao=140 mM`, `ko=2.5 mM`, `cao=2 mM`
- Reversal potentials: `ena=+50 mV`, `ek=-77 mV`, `eca=+132 mV`

However, for any given `Population`, you can customize the initialization of ion concentrations and equilibria by calling `equilibria` and `concentrations`. For example, to initialize intracellular sodium to 15 mM and use the default equilibrium potential:

```python
population = Population(...)
population.concentrations(nai0=15.0)  # sets initial nai to 15
population.equilibria(ena=45.0) # sets ena to +45 mV 
```

Based on the totality of mechanisms in the population and their declared ion usage, Dendra will automatically infer whether equilibria potentials should be 

1. initialized based on intracellular and extracellular concentrations
2. updated based on intracellular and extracellular ion concentrations during simulation. 

The `ion_style` method of the population can be used to override inferred style flags for each ion, which indicate how concentrations and equilibria are handled for that ion in the population. The first argument is the ion name, and the remaining arguments are style flags for concentrations and equilibria. For example, to treat sodium reversal potential as an independent parameter and not update it based on concentrations:

```python
population.ion_style("na", einit=0, eadvance=0)
```

This will ensure that the default initial value for `ena` (which can be overridden via `equilibria`) is used and will not update `ena` during simulation based on changes in `nai` or `nao` (by default, if any mechanisms write to `nai` or `nao`, and any mechanism reads `ena`, `ena` would be updated).